In [2]:
import pandas as pd

In [3]:
pd.set_option('display.max_columns', None)

# in PKL

In [4]:

# ---------------------------------------------------
# PKL CONCAT DATA LOADER MODULE
# ---------------------------------------------------

import pandas as pd
from pathlib import Path
from tqdm import tqdm


def load_pkl_folder(path):
    """
    Load and concatenate all PKL files from a folder (including subfolders).

    Parameters
    ----------
    path : str
        Root directory containing PKL files

    Returns
    -------
    df : pandas.DataFrame
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Folder not found: {path}")

    # ---------------------------------------------------
    # FIND PKL FILES
    # ---------------------------------------------------

    pkl_files = [p for p in path.rglob("*.pkl") if not p.name.startswith("._")]

    if len(pkl_files) == 0:
        raise ValueError("No PKL files found in directory")

    print(f"\nFound {len(pkl_files)} PKL files")

    # ---------------------------------------------------
    # LOAD + CONCAT
    # ---------------------------------------------------

    dfs = []

    for pkl in tqdm(pkl_files, desc="Loading PKLs"):
        try:
            df = pd.read_pickle(pkl)
            df["__source_file"] = pkl.name
            dfs.append(df)
        except Exception as e:
            print(f"Error loading {pkl}: {e}")

    if len(dfs) == 0:
        raise ValueError("No valid PKL files could be loaded")

    df = pd.concat(dfs, ignore_index=True)

    # ---------------------------------------------------
    # INFO
    # ---------------------------------------------------

    print("\nDataset Combined")
    print("---------------------------")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory usage: {mem:.2f} MB")

    return df
path = "/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_9_ML_project/data/raw/03_27_2026_aiff_tracks_data_"

df_pkl = load_pkl_folder(path)


Found 39 PKL files


Loading PKLs: 100%|██████████████████████████████████████████████████████████| 39/39 [00:00<00:00, 574.18it/s]


Dataset Combined
---------------------------
Rows: 2410
Columns: 76
Memory usage: 11.00 MB


In [5]:
len(df_pkl)

2410

# in folders 

In [6]:
def EDA_build_id_df(folder):

    import os
    import re
    import pandas as pd

    rows = []

    # stricter: stop cleanly before next '--'
    pattern = re.compile(r"--id_([^-\s]+(?:-[^-\s]+)*)")

    for root, _, files in os.walk(folder):
        for f in files:

            if f.lower().endswith((".aiff", ".aif")) and not f.startswith(("._", ".DS")):

                match = pattern.search(f)

                if match:
                    id_clean = match.group(1).strip().lower()

                    rows.append({
                        "ID": id_clean,
                        "Path": os.path.join(root, f)
                    })

    df_files = pd.DataFrame(rows)

    print(f"AIFF with ID: {len(df_files)}")

    return df_files

In [7]:
folder = "/Users/yerik/Music/_1_NEW_SOURCE"

df_folders = EDA_build_id_df(folder)

AIFF with ID: 2402


In [8]:
#df_folders[ID]

# merge 

In [9]:
# ---------------------------------
# CLEAN IDS (IMPORTANT)
# ---------------------------------
df_folders['ID_clean'] = df_folders['ID'].astype(str).str.strip().str.lower()
df_pkl['ID_clean']     = df_pkl['ID'].astype(str).str.strip().str.lower()

# ---------------------------------
# MERGE (KEEP ALL df_folders)
# ---------------------------------
df_export = df_folders.merge(
    df_pkl,
    on='ID_clean',
    how='left',
    suffixes=('_folders', '_pkl')
)

# ---------------------------------
# OPTIONAL: CLEAN FINAL STRUCTURE
# ---------------------------------
# keep original ID + Path + all pkl info
df_export = df_export.drop(columns=['ID_clean'])

In [10]:
len(df_export)

2402

In [11]:
df_export.head(3)

,ID_folders,Path_folders,Path_pkl,temp_id,file_name,Extension,dur_seconds,dur_min,sr,bit_depth,bit_rate,channels,file_size,file_size_human,num_frames,error,ms_lufs,ms_LUFS_code,id_cat_lufs,mean_bpm,std_bpm,min_bpm,max_bpm,variation_percentage,dominant_bpm,bpm_consistency,bpm_consistency_cat,title,title_file,artist,artist_file,LABEL,label_file,genre,genre_file,rel_year,rel_year_file,KEY,key_file,mix_name,remixer,remix,date_purchased,rel_date,rel_date_file,key_dj,key_music,status,Relative_Key,Key_Up,Key_Down,Jaw_s_Mix,Mood_Shifter,ID_pkl,comment,Path_jpg_album,Path_jpg_key,Path_jpg_clip,Path_png_bar_dyn,ms_LUFS_norm,Path_png_bar_lufs,Spectral_Bandwidth,Spectral_Flatness,HEX_shape_texture,Path_png_bar_text,spec_centroid_hz,centroid_color,centroid_desc,Path_png_bar_centroid,Path_csv_freq,Path_png_dr,Path_png_id_and_key,year_written_id3,bought_year,lufs_pct,re_name,audio_hash,__source_file
0,t1-12306,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__...,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__...,TRkw_Functional_Anger__ARkw_Rick_Wade_MXkw_Ori...,TRkw_Functional_Anger__ARkw_Rick_Wade_MXkw_Ori...,.aiff,393.799206,6.56332,44100,16,None,2,70378472,67.12 MB,17366545,None,-11.53,T,T,123.426649,2.764799,123.046875,143.554688,2.240034,123,98.148148,D_0,Functional Anger Original Mix,ID3TAGS,Rick Wade,ID3TAGS,Phonogramme,ID3TAGS,Deep House,ID3TAGS,2025,ID3TAGS,Emin,ID3TAGS,Original_Mix,Original Mix,O,2026-01-23,2025-11-10,ID3TAGS,9A,Emin,Success: TKEY updated,9B,10A,8A,4A,12B,t1-12306,0LUFS--T-87%--0Emin_123BPM--id_t1-12306,images/cover_1_t1-12306.jpg,images/key_plott1-12306.png,images/clip_plott1-12306.jpg,images/bpm_bar_dyn_t1-12306.png,76.923077,images/lufs_bar_dyn_t1-12306.png,3800.914308,0.008291,#005EFF,images/text_shape_t1-12306.png,3490.022323,#808000,"Tense, decaying",images/centroid_donut_t1-12306.png,tables/table_freq_t1-12306.csv,images/dbs_plot_t1-12306.png,images/key_and_id_t1-12306.png,Unsupported,2026,87,dylu_0T[26]-123BPM-9A_Emin--id_t1-12306---Deep...,622ea4787b16ecd93da93d52c6febda8c780942622aff8...,df_t6t_final.pkl
1,t1-12300,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__...,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__...,"TRkw_Naked_feat._Aaron_Carl_ARkw_Aaron-Carl,_D...","TRkw_Naked_feat._Aaron_Carl_ARkw_Aaron-Carl,_D...",.aiff,469.746667,7.829111,44100,16,None,2,82900338,79.06 MB,20715828,None,-11.03,T,T,123.046875,0.000000,123.046875,123.046875,0.000000,123,100.000000,D_0,Naked feat Aaron Carl Original Mix,ID3TAGS,Aaron Carl Dyed Soundorom,ID3TAGS,CDR (Crosstown Digital Rebels),ID3TAGS,Deep House,ID3TAGS,2009,ID3TAGS,Gmin,ID3TAGS,Original_Mix,Original Mix,O,2026-01-23,2009-01-26,ID3TAGS,6A,Gmin,Success: TKEY updated,6B,7A,5A,1A,9B,t1-12300,0LUFS--T-87%--0Gmin_123BPM--id_t1-12300,images/cover_1_t1-12300.jpg,images/key_plott1-12300.png,images/clip_plott1-12300.jpg,images/bpm_bar_dyn_t1-12300.png,76.923077,images/lufs_bar_dyn_t1-12300.png,4725.746003,0.032783,#113C70,images/text_shape_t1-12300.png,5344.390429,#FFFF99,Dreamy & bright,images/centroid_donut_t1-12300.png,tables/table_freq_t1-12300.csv,images/dbs_plot_t1-12300.png,images/key_and_id_t1-12300.png,Unsupported,2026,87,dylu_0T[26]-123BPM-6A_Gmin--id_t1-12300---Deep...,71888cd188d034fc921f1b4d8e4bc3961373e5815f4cc7...,df_t6t_final.pkl
2,t1-12304,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__...,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__...,TRkw_Beautiful_Eva_ARkw_Dyed_Soundorom_MXkw_Or...,TRkw_Beautiful_Eva_ARkw_Dyed_Soundorom_MXkw_Or...,.aiff,386.573333,6.442889,44100,16,None,2,68222900,65.06 MB,17047884,None,-11.24,T,T,119.603689,15.284070,83.354335,126.048018,12.778928,126,84.905660,D_6,Beautiful Eva Original Mix,ID3TAGS,Dyed Soundorom,ID3TAGS,CDR (Crosstown Digital Rebels),ID3TAGS,Deep House,ID3TAGS,2009,ID3TAGS,Fmaj,ID3TAGS,Original_Mix,Original Mix,O,2026-01-23,2009-01-26,ID3TAGS,7B,Fmaj,Success: TKEY updated,7A,8B,6B,2B,4A,t1-12304,6LUFS--T-87%--6Fmaj_126BPM--id_t1-12304,images/cover_1_t1-12304.jpg,images/key_plott1-12304.png,images/clip_plott1-12304

# modify var names clean for a new df 

In [12]:
#df_export.columns.tolist()

In [14]:
cols_keep = ['ID_folders',
 'Path_folders',
 'Extension',
 'dur_seconds',
 'dur_min',
 'sr',
 'bit_depth',
#'bit_rate',
 'channels',
 'file_size',
 'file_size_human',
 'num_frames',
 #'error',
 'ms_lufs',
 #'ms_LUFS_code', missing values due to larger lufs
 'id_cat_lufs',
 'mean_bpm',
 'std_bpm',
 'min_bpm',
 'max_bpm',
 'variation_percentage',
 'dominant_bpm',
 'bpm_consistency',
 'bpm_consistency_cat',
 'title',
 'title_file',
 'artist',
 'artist_file',
 'LABEL',
 'label_file',
 'genre',
 'genre_file',
 'rel_year',
 'rel_year_file',
 'KEY',
 'key_file',
 'mix_name',
 'remixer',
 'remix',
 'date_purchased',
 'rel_date',
 'rel_date_file',
 'key_dj',
 'key_music',
 'Relative_Key',
 'Key_Up',
 'Key_Down',
 'Jaw_s_Mix',
 'Mood_Shifter',
 'comment',
 'ms_LUFS_norm',
 'Spectral_Bandwidth',
 'Spectral_Flatness',
 'HEX_shape_texture',
 'spec_centroid_hz',
 'centroid_color',
 'centroid_desc',
 'year_written_id3',
 'bought_year',
 'lufs_pct',
 'audio_hash',
 '__source_file']

In [15]:
df_export = df_export[[c for c in cols_keep if c in df_export.columns]]

In [16]:
pd.set_option('display.max_columns', None)

df_export.head(3)

,ID_folders,Path_folders,Extension,dur_seconds,dur_min,sr,bit_depth,channels,file_size,file_size_human,num_frames,ms_lufs,id_cat_lufs,mean_bpm,std_bpm,min_bpm,max_bpm,variation_percentage,dominant_bpm,bpm_consistency,bpm_consistency_cat,title,title_file,artist,artist_file,LABEL,label_file,genre,genre_file,rel_year,rel_year_file,KEY,key_file,mix_name,remixer,remix,date_purchased,rel_date,rel_date_file,key_dj,key_music,Relative_Key,Key_Up,Key_Down,Jaw_s_Mix,Mood_Shifter,comment,ms_LUFS_norm,Spectral_Bandwidth,Spectral_Flatness,HEX_shape_texture,spec_centroid_hz,centroid_color,centroid_desc,year_written_id3,bought_year,lufs_pct,audio_hash,__source_file
0,t1-12306,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__...,.aiff,393.799206,6.56332,44100,16,2,70378472,67.12 MB,17366545,-11.53,T,123.426649,2.764799,123.046875,143.554688,2.240034,123,98.148148,D_0,Functional Anger Original Mix,ID3TAGS,Rick Wade,ID3TAGS,Phonogramme,ID3TAGS,Deep House,ID3TAGS,2025,ID3TAGS,Emin,ID3TAGS,Original_Mix,Original Mix,O,2026-01-23,2025-11-10,ID3TAGS,9A,Emin,9B,10A,8A,4A,12B,0LUFS--T-87%--0Emin_123BPM--id_t1-12306,76.923077,3800.914308,0.008291,#005EFF,3490.022323,#808000,"Tense, decaying",Unsupported,2026,87,622ea4787b16ecd93da93d52c6febda8c780942622aff8...,df_t6t_final.pkl
1,t1-12300,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__...,.aiff,469.746667,7.829111,44100,16,2,82900338,79.06 MB,20715828,-11.03,T,123.046875,0.000000,123.046875,123.046875,0.000000,123,100.000000,D_0,Naked feat Aaron Carl Original Mix,ID3TAGS,Aaron Carl Dyed Soundorom,ID3TAGS,CDR (Crosstown Digital Rebels),ID3TAGS,Deep House,ID3TAGS,2009,ID3TAGS,Gmin,ID3TAGS,Original_Mix,Original Mix,O,2026-01-23,2009-01-26,ID3TAGS,6A,Gmin,6B,7A,5A,1A,9B,0LUFS--T-87%--0Gmin_123BPM--id_t1-12300,76.923077,4725.746003,0.032783,#113C70,5344.390429,#FFFF99,Dreamy & bright,Unsupported,2026,87,71888cd188d034fc921f1b4d8e4bc3961373e5815f4cc7...,df_t6t_final.pkl
2,t1-12304,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__...,.aiff,386.573333,6.442889,44100,16,2,68222900,65.06 MB,17047884,-11.24,T,119.603689,15.284070,83.354335,126.048018,12.778928,126,84.905660,D_6,Beautiful Eva Original Mix,ID3TAGS,Dyed Soundorom,ID3TAGS,CDR (Crosstown Digital Rebels),ID3TAGS,Deep House,ID3TAGS,2009,ID3TAGS,Fmaj,ID3TAGS,Original_Mix,Original Mix,O,2026-01-23,2009-01-26,ID3TAGS,7B,Fmaj,7A,8B,6B,2B,4A,6LUFS--T-87%--6Fmaj_126BPM--id_t1-12304,76.923077,3003.322452,0.005540,#005EFF,2639.947662,#3C3B6E,Dubby twilight,Unsupported,2026,87,ed45bd1d38502a22a69f1adff5870f477195eb67336f4e...,df_t6t_final.pkl


# rename 

In [17]:
df_export = df_export.rename(columns={'ID_folders': 'ID'})
df_export = df_export.rename(columns={'Path_folders': 'Path'})

# last check , check same id in path than ID 

In [18]:
matches = df_export.apply(
    lambda x: str(x['ID']) in str(x['Path']),
    axis=1
)

count_match = matches.sum()

print(f"IDs found inside Path: {count_match} / {len(df_export)}")

IDs found inside Path: 2402 / 2402


# export df

In [19]:
import os

path_name = '/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_9_ML_project/data/processed/df_03_27_2026_aiff_tracks_data.pkl'

df_export.to_pickle(path_name)

# read 

In [ ]:
path_name = '/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_9_ML_project/data/processed/df_03_27_2026_aiff_tracks_data.pkl'

df_export = pd.read_pickle(path_name)